# Skyra TSER Recon

### 1. Import Libraries

In [1]:
from recon_utils import *
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib qt

/home/hans/Documents/TSER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Load dataset

In [2]:
datafile = 'meas_MID00293_FID02694_461c_std_round_4_6'
data = load_TSER('Data/CMIF/Oct7/' + datafile + '.dat', 128, 400)

pymapVBVD version 0.7.0
Software version: VD


### 3. Fourier Transform to get Image

In [3]:
data_sa = transform_and_scale(data) #points, slices, echoes

In [4]:
print(np.shape(data_sa))

(122, 3, 400)


### 4. Plot 3D data for one slice

In [5]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.view_init(elev=30, azim=-26, roll=0)

distance = np.arange(np.shape(data_sa)[0])
colors = plt.cm.viridis(np.linspace(0, 1, 400))

for i in range(400):
    ax.plot(i, distance, np.abs(data_sa[:,2,i]),color=colors[i], lw=1) #every point, slice 0, echo 'i'

plt.show()

### 5. Plot Slice Cross Section

In [6]:
plt.figure()
plt.plot(np.abs(data_sa[:,2,5:10]))
plt.legend([f'Slice {n}' for n in range(np.shape(data_sa)[1])])
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 5.5 Plot Plate Map

In [7]:
fig, ax = plt.subplots(1,1)
ax.imshow(np.log(np.fliplr(np.flipud(np.abs(data_sa[:,:,50].T)))), aspect='auto') #Every point, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 6. Plot Decay for One Point

In [8]:
point = 21
echoes = 400
te = 10e-3
plt.figure()
time = np.arange(echoes)*te
plt.plot(time, np.abs(data_sa[24,0,:])/np.max(np.abs(data_sa[24,0,10:]))) #Point 52, slice 0, every echo
plt.plot(time, np.abs(data_sa[55,0,:])/np.max(np.abs(data_sa[55,0,10:]))) #Point 52, slice 0, every echo

print(get_t2(np.abs(data_sa[63,0,:]), te=10e-3))
# print(get_t2(np.abs(data_sa[14,2,:]), te=10e-3))
plt.xlabel("Distance (mm)")
plt.ylabel("Amplitude")
plt.show()

0.33315526706423665


### 7. Process to an Array, Save

In [9]:
well_t2s = process_to_array(data_sa, save=True, filename=('' + datafile + '_t2data.csv'))
print(well_t2s)

[[1.55143023 2.42596554        nan        nan        nan        nan
         nan 2.64518517 2.39299999 0.99945511        nan        nan]
 [1.6955672  0.36147736        nan        nan        nan        nan
         nan        nan 1.97577799        nan 0.56830216 1.65200823]
 [2.05676272 1.05005033        nan 0.96892152 2.61505639 0.32645478
  1.93687697        nan        nan 0.98068103        nan        nan]]


In [10]:
plt.figure()
plate = plt.imshow(well_t2s)
ax = plt.gca()
cbar = plt.colorbar(plate)
cbar.set_label('$T_2$ (s)')

ax.set_xticks(np.arange(-.5, 12, 1), minor=True)
ax.set_yticks(np.arange(-.5, 2.5, 1), minor=True)
ax.set_xticks(range(0,12))
ax.set_xticklabels(np.arange(1,13))
yticks = [chr(65+i) for i in range(len(well_t2s))]
ax.set_yticks(range(len(well_t2s)))
ax.set_yticklabels(yticks)
ax.xaxis.tick_top()

ax.grid(which='minor', color='grey', linestyle='-', linewidth=1)
ax.tick_params(which='minor', bottom=False, left=False)

plt.show()